# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from mistralai.client import Mistral
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

mistral_api_key = os.getenv('MISTRAL_API_KEY')
if mistral_api_key:
    print(f"Mistral API Key exists and begins {mistral_api_key[:8]}")
else:
    print("Mistral API Key not set")
    
MODEL = "mistral-small-latest"
mistral = Mistral(api_key=mistral_api_key)

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


Mistral API Key exists and begins qsF5gGNR


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = mistral.chat.stream(model=MODEL, messages=messages)

    response= ''

    for chunk in stream: 
        response += chunk.data.choices[0].delta.content or ''
        yield response

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [5]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "not known, just make up a price in USD. Like 200$")
    return f"The price of a ticket to {destination_city} is {price}"


In [6]:
get_ticket_price("Amsterdam")

Tool called for city Amsterdam


'The price of a ticket to Amsterdam is not known, just make up a price in USD. Like 200$'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [10]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    rel_message = system_message
    rel_message += 'If you get a reply of the tool call, follow the tool call'

    messages = [{"role": "system", "content": rel_message}] + history + [{"role": "user", "content": message}]
    response = mistral.chat.complete(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        for message in messages:
            print(message)
        response = mistral.chat.complete(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [11]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = mistral.chat.complete(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = mistral.chat.complete(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [17]:
import sqlite3


In [18]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [19]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [20]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [93]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        return f"The ticket price of {city} has been updated to {price}"

In [94]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [95]:
set_ticket_price("Tokyo",300)

'The ticket price of Tokyo has been updated to 300'

In [96]:
# Writing the update function in Json:

update_price = {
  "name": "update_price",
  "description": "Updates the price to a destination city.",
  "parameters": {
    "type": "object",
    "properties": {
      "destination_city": {
        "type": "string",
        "description": "The city that the customer wants to get updated"
      },
      "price": {
        "type": "number",
        "description": "The price of the ticket"
      }
    },
    "required": ["destination_city", "price"],
    "additionalProperties": False
  }
}

In [97]:
tools = [{"type": "function", "function": price_function},
         {"type":"function", "function":update_price}]

In [98]:
 arguments={"destination_city": "Japan", "price": 300}

In [99]:
tool_arguments = {'get_ticket_price':
                  {"func":get_ticket_price,
                   'city':arguments.get('destination_city')},
                   'update_price':{"func":set_ticket_price,
                                   "city":arguments.get('destination_city'),
                                   "price":arguments.get('price')}}

In [ ]:
# Updating the handle:


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        #  Getting the arguments
        arguments = json.loads(tool_call.function.arguments)


        reply = tool_arguments[tool_call.function.name]["func"](*arguments.values()) #This is a new helpfull lesson. the * unpacks the values directly, regardless of the size. As long as the order is logical, this works!
        print(reply)

        responses.append({
                "role": "tool",
                "content": reply,
                "tool_call_id": tool_call.id})
    return responses

In [101]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = mistral.chat.complete(model=MODEL, messages=messages, tools=tools)
    print(response)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = mistral.chat.complete(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

id='a66bbdef61244a028e63bf59b881e230' object='chat.completion' model='mistral-small-latest' usage=UsageInfo(prompt_tokens=305, completion_tokens=20, total_tokens=325, prompt_audio_seconds=Unset(), prompt_tokens_details={'cached_tokens': 0}) created=1780837467 choices=[ChatCompletionChoice(index=0, finish_reason='tool_calls', message=AssistantMessage(role='assistant', content='', tool_calls=[ToolCall(function=FunctionCall(name='update_price', arguments='{"destination_city": "Paris", "price": 200}'), id='1TrHzHebu', type='function', index=0)], prefix=False), messages=None)]
The ticket price of Paris has been updated to 200


In [102]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


id='b485256723634aa0b956699feab84ecc' object='chat.completion' model='mistral-small-latest' usage=UsageInfo(prompt_tokens=250, completion_tokens=10, total_tokens=260, prompt_audio_seconds=Unset(), prompt_tokens_details={'cached_tokens': 0}) created=1780837479 choices=[ChatCompletionChoice(index=0, finish_reason='stop', message=AssistantMessage(role='assistant', content='Hello! How can I assist you today?', tool_calls=None, prefix=False), messages=None)]
id='70904cc9dc7940c2b0905012a2cce2d2' object='chat.completion' model='mistral-small-latest' usage=UsageInfo(prompt_tokens=268, completion_tokens=9, total_tokens=277, prompt_audio_seconds=Unset(), prompt_tokens_details={'cached_tokens': 0}) created=1780837486 choices=[ChatCompletionChoice(index=0, finish_reason='stop', message=AssistantMessage(role='assistant', content='Where would you like to fly to?', tool_calls=None, prefix=False), messages=None)]
id='490c309b4a21434faa9cda112fdbc1cb' object='chat.completion' model='mistral-small-late

## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>